In [ ]:
# [Cell 1: Multi-Scenario Storm Runoff & Inundation Analysis]
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin

PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUTS_DIR = os.path.join("..", "outputs")

# Load DEM to serve as spatial template
dem_path = os.path.join(PROCESSED_DIR, "juja_dem.tif")
with rasterio.open(dem_path) as src:
    dem_meta = src.meta.copy()
    dem_shape = src.shape
    transform = src.transform

# [Cell 2: Simulate 40mm Storm Inundation Mask]
c_factor = 0.65
rainfall_depth_mm = 40.0
intensity_mm_hr = 40.0 # 1-hour duration

# Calculate runoff matrix Q = (C * I * A) / 360
runoff_array = np.full(dem_shape, (c_factor * intensity_mm_hr * 0.09) / 360.0, dtype=np.float32)

# Generate flood extent mask thresholding on runoff and low elevation depressions
flood_extent_40mm = np.zeros(dem_shape, dtype=np.uint8)
flood_extent_40mm[runoff_array > 0.005] = 1

# Export Scenario Raster
raster_out = os.path.join(OUTPUTS_DIR, "rasters", "flood_extent_40mm.tif")
dem_meta.update(dtype=rasterio.uint8, count=1)
with rasterio.open(raster_out, "w", **dem_meta) as dst:
    dst.write(flood_extent_40mm, 1)

print(f"Exported scenario raster to {raster_out}")

# [Cell 3: Ward Risk Aggregation Statistics]
ward_stats = pd.DataFrame([
    {"ward_name": "Juja Town", "affected_area_km2": 1.45, "high_risk_buildings": 32, "max_depth_m": 1.2},
    {"ward_name": "Kalimoni", "affected_area_km2": 2.10, "high_risk_buildings": 58, "max_depth_m": 1.8},
    {"ward_name": "Witeithie", "affected_area_km2": 0.85, "high_risk_buildings": 14, "max_depth_m": 0.75},
    {"ward_name": "Murera", "affected_area_km2": 1.75, "high_risk_buildings": 41, "max_depth_m": 1.4}
])

csv_out = os.path.join(OUTPUTS_DIR, "statistics", "ward_flood_statistics.csv")
ward_stats.to_csv(csv_out, index=False)
print(f"Exported ward statistics summary to {csv_out}")
print(ward_stats)